# 🚀 Multimodal RAG with Docling, Qdrant & Groq

Parse PDFs, index them into **Persistent Qdrant (Cloud or Local)** or **In-Memory**, and perform RAG generation with **Groq**.

### Pipeline Overview:
1. **Parse & Chunk**: Docling (`DocumentConverter` / `HybridChunker`)
2. **Embed**: Dense (`sentence-transformers`) + Sparse (`fastembed` SPLADE)
3. **Vector Store**: Qdrant (Cloud URL / Local `http://localhost:6333` / `:memory:`)
4. **Query & RAG**: Hybrid search + Groq (`llama-3.3-70b-versatile`) Answer Generation

## 📦 1. Install Dependencies

In [1]:
!pip install -q docling docling-core qdrant-client sentence-transformers fastembed openai python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 718.1/718.1 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 286.4/286.4 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.2/79.2 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 74.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.9/323.9 kB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.0/94.0 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 120.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 🔑 2. Set Groq & Qdrant Credentials & Load PDF

In [ ]:
import os
from pathlib import Path

# Credentials setup
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
    QDRANT_URL = ""
    QDRANT_API_KEY = userdata.get("QDRANT_API_KEY")
except Exception:
    GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "gsk_your_groq_api_key_here")
    QDRANT_URL = os.environ.get("QDRANT_URL", "")
    QDRANT_API_KEY = os.environ.get("QDRANT_API_KEY", "")

# Target Collection Name
COLLECTION_NAME = os.environ.get("QDRANT_COLLECTION_NAME", "multimodal_rag_docs")

# # Upload PDF
# try:
#     from google.colab import files
#     print("Upload your PDF document:")
#     uploaded = files.upload()
#     PDF_PATH = list(uploaded.keys())[0]
# except ImportError:
PDF_PATH = "attention_is_all_you_need.pdf"

print(f"Target Document: {PDF_PATH} ({os.path.getsize(PDF_PATH):,} bytes)")
print(f"Target Qdrant Collection: {COLLECTION_NAME}")

Target Document: attention_is_all_you_need.pdf (2,215,244 bytes)
Target Qdrant Collection: multimodal_rag_docs


## ⚙️ 3. Parse Document & Extract Chunks with Docling

In [5]:
from docling.document_converter import DocumentConverter
from docling.chunking import HybridChunker

print("Converting PDF with Docling...")
converter = DocumentConverter()
result = converter.convert(PDF_PATH)
doc = result.document

print(f"Parsed document: '{doc.name}' with {len(doc.pages)} pages.")

# Chunk document using Docling HybridChunker
chunker = HybridChunker()
raw_chunks = list(chunker.chunk(doc))

chunks = []
for idx, chunk in enumerate(raw_chunks):
    text = chunk.text.strip()
    page_no = chunk.meta.doc_items[0].prov[0].page_no if chunk.meta.doc_items else 1
    if text:
        chunks.append({
            "id": idx,
            "text": text,
            "page_number": page_no,
        })

print(f"Extracted {len(chunks)} chunks.")

Converting PDF with Docling...


[INFO] 2026-08-04 03:40:21,955 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-08-04 03:40:21,988 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-08-04 03:40:21,989 [RapidOCR] main.py:63: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-08-04 03:40:22,050 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-08-04 03:40:22,054 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-08-04 03:40:22,055 [RapidOCR] main.py:63: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-08-04 03:40:22,105 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-08-04 03:40:22,174 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/l

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

W0804 03:41:21.262000 2288 torch/_inductor/utils.py:1731] [2/0_1] Not enough SMs to use max_autotune_gemm mode


Parsed document: 'attention_is_all_you_need' with 15 pages.


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (968 > 512). Running this sequence through the model will result in indexing errors


Extracted 67 chunks.


## 🧠 4. Generate Dense & Sparse Embeddings

In [6]:
from sentence_transformers import SentenceTransformer
from fastembed import SparseTextEmbedding

print("Loading embedding models...")
dense_model = SentenceTransformer("BAAI/bge-small-en-v1.5")
sparse_model = SparseTextEmbedding("prithivida/Splade_PP_en_v1")

texts = [c["text"] for c in chunks]

print("Generating dense embeddings...")
dense_embeddings = dense_model.encode(texts, show_progress_bar=True, normalize_embeddings=True)

print("Generating sparse embeddings (SPLADE)...")
sparse_embeddings = list(sparse_model.embed(texts))

print("✅ Embeddings generated successfully.")

Loading embedding models...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Generating dense embeddings...


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Generating sparse embeddings (SPLADE)...
✅ Embeddings generated successfully.


## 🗄️ 5. Index Vectors in Persistent Qdrant (Cloud / Local)

In [7]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, SparseVectorParams, Distance, PointStruct, SparseVector

# Connect to Persistent Qdrant Cloud / Local if QDRANT_URL is set, else fall back to :memory:
if QDRANT_URL and QDRANT_URL.startswith("http"):
    print(f"🌐 Connecting to Persistent Qdrant Cluster at: {QDRANT_URL}")
    client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY or None)
else:
    print("ℹ️ No QDRANT_URL provided. Using in-memory Qdrant instance (:memory:).")
    client = QdrantClient(":memory:")

# Create collection with Hybrid (Dense + Sparse) config
client.recreate_collection(
    collection_name=COLLECTION_NAME,
    vectors_config={"dense": VectorParams(size=384, distance=Distance.COSINE)},
    sparse_vectors_config={"sparse": SparseVectorParams()}
)

# Upsert points
points = []
for i, chunk in enumerate(chunks):
    s_emb = sparse_embeddings[i]
    points.append(PointStruct(
        id=chunk["id"],
        vector={
            "dense": dense_embeddings[i].tolist(),
            "sparse": SparseVector(indices=s_emb.indices.tolist(), values=s_emb.values.tolist())
        },
        payload={
            "text": chunk["text"],
            "page_number": chunk["page_number"]
        }
    ))

client.upsert(collection_name=COLLECTION_NAME, points=points)
print(f"✅ Successfully indexed {len(points)} chunks into Qdrant collection '{COLLECTION_NAME}'.")

🌐 Connecting to Persistent Qdrant Cluster at: https://491a2582-7d00-4644-9809-a3faab7fab8a.eu-west-2-0.aws.cloud.qdrant.io


/tmp/ipykernel_2288/4222218969.py:13: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


✅ Successfully indexed 67 chunks into Qdrant collection 'multimodal_rag_docs'.


## 🔍 6. Multimodal Test Suite & RAG Evaluation

In [8]:
from qdrant_client.models import Prefetch, FusionQuery, Fusion
from openai import OpenAI

def hybrid_search(query: str, top_k: int = 3):
    q_dense = dense_model.encode(query, normalize_embeddings=True).tolist()
    q_sparse_obj = list(sparse_model.embed([query]))[0]
    q_sparse = SparseVector(indices=q_sparse_obj.indices.tolist(), values=q_sparse_obj.values.tolist())

    res = client.query_points(
        collection_name=COLLECTION_NAME,
        prefetch=[
            Prefetch(query=q_dense, using="dense", limit=top_k * 2),
            Prefetch(query=q_sparse, using="sparse", limit=top_k * 2),
        ],
        query=FusionQuery(fusion=Fusion.RRF),
        limit=top_k,
        with_payload=True,
    )
    return res.points

TEST_QUERIES = [
    "What are the BLEU scores for Transformer (big) on the WMT 2014 English-to-German and English-to-French translation tasks?",
    "What is the mathematical formula for Scaled Dot-Product Attention, including the scaling factor sqrt(d_k)?",
    "Describe the visual architecture of the Transformer model from Figure 1, detailing the Encoder and Decoder sub-layers.",
    "Why is Scaled Dot-Product Attention divided by sqrt(d_k) when d_k is large?"
]

groq_client = None
if GROQ_API_KEY and not GROQ_API_KEY.startswith("gsk_your_"):
    groq_client = OpenAI(api_key=GROQ_API_KEY, base_url="https://api.groq.com/openai/v1")

for idx, query in enumerate(TEST_QUERIES, 1):
    print(f"\n--- [Query {idx}] {query} ---")
    results = hybrid_search(query, top_k=3)
    passages = []
    for r in results:
        p_no = r.payload['page_number']
        txt = r.payload['text']
        passages.append(f"(Page {p_no}) {txt}")
        print(f" • [Page {p_no}] Score: {r.score:.4f} | Preview: {txt[:120]}...")

    if groq_client:
        resp = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": "You are a scientific assistant. Answer concisely using provided context."},
                {"role": "user", "content": f"Context:\n" + "\n\n".join(passages) + f"\n\nQuestion: {query}"}
            ]
        )
        print("\n🤖 Groq Answer:")
        print(resp.choices[0].message.content)


--- [Query 1] What are the BLEU scores for Transformer (big) on the WMT 2014 English-to-German and English-to-French translation tasks? ---
 • [Page 8] Score: 1.0000 | Preview: On the WMT 2014 English-to-German translation task, the big transformer model (Transformer (big) in Table 2) outperforms...
 • [Page 8] Score: 0.5333 | Preview: Table 2: The Transformer achieves better BLEU scores than previous state-of-the-art models on the English-to-German and ...
 • [Page 1] Score: 0.4167 | Preview: The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include a...

🤖 Groq Answer:
The BLEU scores for Transformer (big) are: 
- English-to-German: 28.4
- English-to-French: 41.0

--- [Query 2] What is the mathematical formula for Scaled Dot-Product Attention, including the scaling factor sqrt(d_k)? ---
 • [Page 4] Score: 1.0000 | Preview: We call our particular attention "Scaled Dot-Product Attention" (Figure 2). The input consists of que